# 03 — Technical Indicators

**Goal:** Enrich the raw OHLCV data with technical analysis signals that will feed the anomaly-detection model.

All indicators are implemented from scratch using **pure pandas** — no external TA library required. This is intentional: showing the underlying maths is more instructive (and more reliable across environments) than calling a black-box function.

| Indicator | Method | Purpose |
|---|---|---|
| SMA50, SMA200 | `rolling(n).mean()` | Trend direction; Golden/Death Cross events |
| EMA20 | `ewm(span=20)` | Responsive trend proxy |
| RSI14 | Wilder smoothing | Momentum oscillator; overbought/oversold flags |
| Bollinger Bands BB20 | `rolling(20).mean() ± 2σ` | Volatility bands; breakout flags |
| Rolling Z-score | `(r − μ̄) / σ̄` | Statistical anomaly flag on daily returns |

**DuckDB** produces cross-ticker event-count summaries via SQL.

In [1]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

DATA_DIR = Path('data')
OUT_DIR  = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)
print('Imports OK')

Imports OK


In [2]:
# ── 2. Load data ───────────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'market_data.csv', parse_dates=['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print(f'Loaded: {df.shape}  rows  |  Tickers: {sorted(df.Ticker.unique())}')

Loaded: (12426, 10)  rows  |  Tickers: ['AAPL', 'AMZN', 'GLD', 'JPM', 'SPY', 'XOM']


## Helper — RSI (Wilder Smoothing)

RSI is computed as:
1. Separate daily changes into gains (Up) and losses (Down)
2. Apply Wilder's smoothing (equivalent to EMA with α = 1/n)
3. RS = avg_gain / avg_loss;  RSI = 100 − 100/(1+RS)

In [3]:
# ── 3. RSI helper ─────────────────────────────────────────────────────────
def compute_rsi(close: pd.Series, length: int = 14) -> pd.Series:
    """Return RSI using Wilder smoothing (exact pandas_ta equivalent)."""
    delta = close.diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)
    # Wilder smoothing = EMA with com = length-1
    avg_gain = gain.ewm(com=length - 1, min_periods=length, adjust=False).mean()
    avg_loss = loss.ewm(com=length - 1, min_periods=length, adjust=False).mean()
    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi

print('RSI helper defined.')

RSI helper defined.


## Compute All Indicators Per Ticker

Each group (ticker) is processed independently so rolling windows never bleed across tickers.

In [4]:
# ── 4. Main indicator loop ─────────────────────────────────────────────────
result_frames = []

for ticker, grp in df.groupby('Ticker'):
    g = grp.copy().sort_values('Date').reset_index(drop=True)
    c = g['Close']

    # ── Moving averages ────────────────────────────────────────────────
    g['SMA50']  = c.rolling(50).mean()
    g['SMA200'] = c.rolling(200).mean()
    g['EMA20']  = c.ewm(span=20, adjust=False).mean()

    # ── Golden / Death Cross ───────────────────────────────────────────
    above = g['SMA50'] > g['SMA200']
    g['GoldenCross'] = (above & ~above.shift(1).fillna(False)).astype(int)
    g['DeathCross']  = (~above & above.shift(1).fillna(True)).astype(int)

    # ── RSI14 ──────────────────────────────────────────────────────────
    g['RSI14']          = compute_rsi(c, 14)
    g['RSI_Overbought'] = (g['RSI14'] > 70).astype(int)
    g['RSI_Oversold']   = (g['RSI14'] < 30).astype(int)

    # ── Bollinger Bands (20-day, 2σ) ──────────────────────────────────
    bb_mid            = c.rolling(20).mean()
    bb_std            = c.rolling(20).std()
    g['BB_Upper']      = bb_mid + 2 * bb_std
    g['BB_Mid']        = bb_mid
    g['BB_Lower']      = bb_mid - 2 * bb_std
    g['BB_Upper_Breach'] = (c > g['BB_Upper']).astype(int)
    g['BB_Lower_Breach'] = (c < g['BB_Lower']).astype(int)

    # ── Rolling 30-day Z-score of DailyReturn ─────────────────────────
    r              = g['DailyReturn']
    rolling_mean   = r.rolling(30).mean()
    rolling_std    = r.rolling(30).std()
    g['ZScore']    = (r - rolling_mean) / rolling_std
    g['is_statistical_anomaly'] = (g['ZScore'].abs() > 3).astype(int)

    result_frames.append(g)

df = pd.concat(result_frames, ignore_index=True).sort_values(['Ticker','Date']).reset_index(drop=True)

print(f'Indicators computed.  Shape: {df.shape}')
print(f'  Golden Cross events : {df.GoldenCross.sum()}')
print(f'  Death  Cross events : {df.DeathCross.sum()}')
print(f'  RSI Overbought days : {df.RSI_Overbought.sum()}')
print(f'  RSI Oversold   days : {df.RSI_Oversold.sum()}')
print(f'  BB Upper breaches   : {df.BB_Upper_Breach.sum()}')
print(f'  BB Lower breaches   : {df.BB_Lower_Breach.sum()}')
print(f'  Z-score anomalies   : {df.is_statistical_anomaly.sum()}')

Indicators computed.  Shape: (12426, 25)
  Golden Cross events : 8133
  Death  Cross events : 35
  RSI Overbought days : 1131
  RSI Oversold   days : 300
  BB Upper breaches   : 771
  BB Lower breaches   : 555
  Z-score anomalies   : 100


In [5]:
# ── 5. Save indicators CSV ─────────────────────────────────────────────────
OUT_CSV = DATA_DIR / 'market_data_indicators.csv'
df.to_csv(OUT_CSV, index=False)
print(f'Saved → {OUT_CSV}  ({OUT_CSV.stat().st_size/1024:.0f} KB)')
print(f'Columns: {df.columns.tolist()}')

Saved → data\market_data_indicators.csv  (3817 KB)
Columns: ['Date', 'Ticker', 'Close', 'Volume', 'High', 'Low', 'DailyReturn', 'LogReturn', 'CumReturn', '^VIX', 'SMA50', 'SMA200', 'EMA20', 'GoldenCross', 'DeathCross', 'RSI14', 'RSI_Overbought', 'RSI_Oversold', 'BB_Upper', 'BB_Mid', 'BB_Lower', 'BB_Upper_Breach', 'BB_Lower_Breach', 'ZScore', 'is_statistical_anomaly']


## DuckDB — Signal Summary Table

One SQL query produces the full cross-ticker breakdown — much more concise than a multi-column pandas GroupBy.

In [6]:
# ── 6. DuckDB — indicator event summary ──────────────────────────────────
con = duckdb.connect()
con.register('indicators', df)

summary = con.execute("""
    SELECT
        Ticker,
        SUM(GoldenCross)            AS golden_crosses,
        SUM(DeathCross)             AS death_crosses,
        SUM(RSI_Overbought)         AS rsi_overbought_days,
        SUM(RSI_Oversold)           AS rsi_oversold_days,
        SUM(BB_Upper_Breach)        AS bb_upper_breaches,
        SUM(BB_Lower_Breach)        AS bb_lower_breaches,
        SUM(is_statistical_anomaly) AS z_score_anomalies
    FROM indicators
    GROUP BY Ticker
    ORDER BY Ticker
""").df()

print('=== Technical Indicator Summary (DuckDB) ===')
print(summary.to_string(index=False))

=== Technical Indicator Summary (DuckDB) ===
Ticker  golden_crosses  death_crosses  rsi_overbought_days  rsi_oversold_days  bb_upper_breaches  bb_lower_breaches  z_score_anomalies
  AAPL          1417.0            6.0                264.0               50.0              138.0               90.0               20.0
  AMZN          1291.0            7.0                132.0               49.0              137.0               89.0               17.0
   GLD          1432.0            6.0                211.0               52.0              178.0               87.0               14.0
   JPM          1390.0            5.0                212.0               36.0              110.0               88.0               16.0
   SPY          1470.0            5.0                182.0               36.0               88.0              101.0               23.0
   XOM          1133.0            6.0                130.0               77.0              120.0              100.0               10.0


In [7]:
# ── 7. DuckDB — all death cross events ───────────────────────────────────
cross_events = con.execute("""
    SELECT
        Date, Ticker,
        ROUND(Close,  2)  AS Close,
        ROUND(SMA50,  2)  AS SMA50,
        ROUND(SMA200, 2)  AS SMA200
    FROM indicators
    WHERE DeathCross = 1
    ORDER BY Date
""").df()

print('\n=== All Death Cross Events ===')
print(cross_events.to_string(index=False))


=== All Death Cross Events ===
      Date Ticker  Close  SMA50  SMA200
2018-01-02    XOM  58.58    NaN     NaN
2018-01-02   AMZN  59.45    NaN     NaN
2018-01-02    GLD 125.15    NaN     NaN
2018-01-02   AAPL  40.30    NaN     NaN
2018-01-02    JPM  86.34    NaN     NaN
2018-01-02    SPY 236.56    NaN     NaN
2018-11-23    JPM  87.26  90.39   90.42
2018-12-12    SPY 236.70 244.22  244.63
2018-12-12   AMZN  83.18  84.84   85.19
2018-12-20    XOM  49.24  56.19   56.37
2018-12-21   AAPL  35.80  45.69   45.80
2019-06-19    XOM  55.27  56.19   56.20
2019-10-10   AMZN  86.01  89.24   89.32
2020-03-27    JPM  77.38 101.42  101.57
2020-03-31    SPY 236.93 274.28  274.97
2021-02-17    GLD 166.33 173.97  174.06
2021-04-19   AMZN 168.60 159.04  159.10
2021-08-06    GLD 164.64 170.46  170.58
2022-01-25   AMZN 139.99 168.77  169.27
2022-01-27    GLD 167.60 168.74  168.77
2022-02-01    JPM 136.29 142.13  142.21
2022-03-16    SPY 410.94 418.29  418.89
2022-06-03   AAPL 142.65 155.96  156.11
2022-07-

In [8]:
# ── 8. DuckDB — top Z-score anomaly days ─────────────────────────────────
top_z = con.execute("""
    SELECT
        Date, Ticker,
        ROUND(Close, 2)           AS Close,
        ROUND(DailyReturn*100, 2) AS DailyRet_Pct,
        ROUND(ZScore, 2)          AS ZScore,
        ROUND(\"^VIX\", 1)          AS VIX
    FROM indicators
    WHERE is_statistical_anomaly = 1
    ORDER BY ABS(ZScore) DESC
    LIMIT 20
""").df()

print('\n=== Top 20 Z-Score Anomaly Days ===')
print(top_z.to_string(index=False))


=== Top 20 Z-Score Anomaly Days ===
      Date Ticker  Close  DailyRet_Pct  ZScore  VIX
2018-10-10    SPY 248.15         -3.17   -4.49 23.0
2024-11-06    JPM 240.85         11.54    4.48 16.3
2024-04-12    JPM 176.14         -6.47   -4.27 17.3
2020-08-11    GLD 179.94         -5.37   -4.20 24.0
2020-11-09    JPM 102.21         13.54    4.20 25.8
2026-01-30    GLD 444.95        -10.27   -4.19 17.4
2025-08-01   AMZN 214.75         -8.27   -4.19 20.4
2025-10-10    SPY 649.32         -2.70   -4.06 21.7
2018-08-01   AAPL  47.52          5.89    4.03 13.1
2020-09-03    SPY 318.89         -3.44   -4.00 33.6
2025-04-09    SPY 542.40         10.50    3.98 33.6
2021-11-26    SPY 431.45         -2.23   -3.96 28.6
2022-02-04   AMZN 157.64         13.54    3.90 23.2
2024-02-02   AMZN 171.81          7.87    3.90 13.9
2025-04-09   AAPL 197.99         15.33    3.88 33.6
2025-10-21    GLD 377.24         -6.43   -3.88 17.9
2022-04-29   AMZN 124.28        -14.05   -3.87 33.4
2020-07-31   AAPL 102.98   

In [9]:
print('\n✓ 03_indicators complete')
print(f'  → {OUT_CSV}  ({OUT_CSV.stat().st_size/1024:.0f} KB)')


✓ 03_indicators complete
  → data\market_data_indicators.csv  (3817 KB)
